# PYNQ V03 错误与恢复门禁

本 Notebook 会主动注入超长请求、半帧、短 deadline、重复 request_id、错误 epoch 和 RX overflow。
每个故障后都会 reset SoC、确认 epoch 改变，并重新运行 SW/HW/ISE KAT。
测试会重置 PL 内的 RV32 SoC，但不会重新生成或覆盖 bit/hwh/firmware。

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import contextlib
import io
import json
import os
import platform
import subprocess
import sys
import pynq

repo = Path(os.environ.get("ASCON_REPO", "/home/xilinx/jupyter_notebooks/amd_fpga_2026")).resolve()
assert (repo / "src/pynq/board_error_gate.py").is_file(), "请同步仓库或设置 ASCON_REPO"
os.chdir(str(repo))
sys.path.insert(0, str(repo))

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = repo / "data/pynq" / ("error_gate_" + run_id)
run_dir.mkdir(parents=True, exist_ok=False)
identity = {
    "generated_local": datetime.now().isoformat(timespec="seconds"),
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=str(repo), text=True
    ).strip(),
    "pynq_version": pynq.__version__,
    "python_version": platform.python_version(),
    "kernel": platform.platform(),
}
(run_dir / "identity.json").write_text(
    json.dumps(identity, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(identity, indent=2))

{
  "generated_local": "2026-10-29T13:40:23",
  "generated_utc": "2026-10-29T13:40:23+00:00",
  "git_commit": "ac9143d5c6acc9e7ac61f907a7e802eb6b35cddd",
  "pynq_version": "2.7.0",
  "python_version": "3.8.2",
  "kernel": "Linux-5.4.0-xilinx-v2020.2-armv7l-with-glibc2.29"
}


## 执行错误门禁

默认每种模式执行 50 次合法加密/解密往返。失败仍会保存 JSON，然后在最后一个单元抛出断言。

In [2]:
from src.pynq.board_error_gate import run_error_gate

gate_log = io.StringIO()
with contextlib.redirect_stdout(gate_log), contextlib.redirect_stderr(gate_log):
    gate = run_error_gate(
        "board/vivado_zynq7000_xc7z020_pynq/release",
        stress_per_mode=50,
        kat_count=529,
    )

(run_dir / "pynq_board_error_gate.json").write_text(
    json.dumps(gate, indent=2, default=str) + "\n", encoding="utf-8"
)
(run_dir / "pynq_board_error_gate.log").write_text(
    gate_log.getvalue(), encoding="utf-8"
)
print(gate_log.getvalue())
print("error-gate evidence saved to", run_dir)

V03 ERROR GATE START: release=board/vivado_zynq7000_xc7z020_pynq/release kat=529 stress_per_mode=50
START initial_health
PASS  initial_health
START over_capacity_data_17B
PASS  over_capacity_data_17B
START over_capacity_ad_17B
PASS  over_capacity_ad_17B
START half_frame_requires_reset
PASS  half_frame_requires_reset
START deadline_timeout_requires_recovery
PASS  deadline_timeout_requires_recovery
START duplicate_request_id_rejected
PASS  duplicate_request_id_rejected
START stale_epoch_rejected
PASS  stale_epoch_rejected
START rx_fifo_overflow_detected
PASS  rx_fifo_overflow_detected
START repeated_requests
PASS  repeated_requests
V03 ERROR GATE PASS: checks=9 resets=7

error-gate evidence saved to /home/xilinx/jupyter_notebooks/amd_fpga_2026/data/pynq/error_gate_20261029_134023


In [3]:
for item in gate["checks"]:
    print("{:<40} {}".format(item["name"], item["status"]))
    if item["status"] != "PASS":
        print("  ", item.get("error_type"), item.get("error"))

assert gate["board_validation"] == "PASS", gate["checks"]

initial_health                           PASS
over_capacity_data_17B                   PASS
over_capacity_ad_17B                     PASS
half_frame_requires_reset                PASS
deadline_timeout_requires_recovery       PASS
duplicate_request_id_rejected            PASS
stale_epoch_rejected                     PASS
rx_fifo_overflow_detected                PASS
repeated_requests                        PASS
